In [ ]:
import pandas as pd

# Load the dataset
file_path = './data.csv'
data = pd.read_csv(file_path)

In [ ]:
# Convert datetime columns to proper datetime format if they exist in the dataset
if 'tpep_pickup_datetime' in data.columns and 'tpep_dropoff_datetime' in data.columns:
    data['tpep_pickup_datetime'] = pd.to_datetime(data['tpep_pickup_datetime'])
    data['tpep_dropoff_datetime'] = pd.to_datetime(data['tpep_dropoff_datetime'])
    

In [ ]:
# QN 1: Extract all trips with trip_distance larger than 50
trips_large_distance = data[data['trip_distance'] > 50]
print(trips_large_distance.head())

In [ ]:
# QN 2: Extract all trips where payment_type is missing
trips_missing_payment = data[data['payment_type'].isnull()]
print(trips_missing_payment)

In [ ]:
# QN 3: For each (PULocationID, DOLocationID) pair, determine the number of trips
trips_per_location_pair = data.groupby(['PULocationID', 'DOLocationID']).size().reset_index(name='trip_count')
print(trips_per_location_pair.head())

In [ ]:
# QN 4: Save rows with missing VendorID, passenger_count, store_and_fwd_flag, payment_type
columns_to_check = ['VendorID', 'passenger_count', 'store_and_fwd_flag', 'payment_type']
bad_rows = data[data[columns_to_check].isnull().any(axis=1)]
bad_rows.to_csv('bad_rows.csv', index=False)  # Save bad rows to a new file
data_cleaned = data.drop(bad_rows.index)      # Remove bad rows from the original dataframe

In [ ]:
print(f"Number of bad rows: {len(bad_rows)}")
print(bad_rows)

In [ ]:
print(f"Number of rows after cleaning: {len(data_cleaned)}")
print(data_cleaned.head())

In [ ]:
# QN 5: Add a duration column storing how long each trip has taken
data_cleaned['duration'] = (data_cleaned['tpep_dropoff_datetime'] - data_cleaned['tpep_pickup_datetime']).dt.total_seconds() /60
print(data_cleaned[['tpep_pickup_datetime', 'tpep_dropoff_datetime', 'duration']].head())

In [ ]:

# QN 6: For each pickup location, determine how many trips have started there
trips_per_pickup = data_cleaned['PULocationID'].value_counts().reset_index(name='trip_count').sort_values(by='PULocationID', ascending=True)
trips_per_pickup.columns = ['PULocationID', 'trip_count']
print(trips_per_pickup)

In [ ]:
# QN 6: For each pickup location, determine how many trips have started there
pickup_trip_counts = data_cleaned.groupby('PULocationID').size().reset_index(name='trip_count')
print(pickup_trip_counts)

In [ ]:
# QN 7: Cluster the pickup time of the day into 30-minute intervals
data_cleaned['pickup_interval'] = data_cleaned['tpep_pickup_datetime'].dt.floor('30min')
print(data_cleaned[['tpep_pickup_datetime', 'pickup_interval']].head())

In [ ]:
# QN 8: For each interval, determine the average number of passengers and the average fare amount
interval_stats = data_cleaned.groupby('pickup_interval').agg({
    'passenger_count': 'mean',
    'fare_amount': 'mean'
}).reset_index()
print(interval_stats.head())

In [ ]:
# QN 9: For each payment type and each interval, determine the average fare amount
fare_by_payment_interval = data_cleaned.groupby(['payment_type', 'pickup_interval'])['fare_amount'].mean().reset_index()
print(fare_by_payment_interval.head())

In [ ]:
# QN 10: For each payment type, determine the interval when the average fare amount is maximum
max_fare_interval = fare_by_payment_interval.loc[fare_by_payment_interval.groupby('payment_type')['fare_amount'].idxmax()]
print(max_fare_interval.head())

In [ ]:
# QN 11: For each payment type, determine the interval when the tip-to-fare ratio is maximum
data_cleaned['tip_fare_ratio'] = data_cleaned['tip_amount'] / data_cleaned['fare_amount']
max_ratio_interval = data_cleaned.groupby(['payment_type', 'pickup_interval'])['tip_fare_ratio'].mean().reset_index()
# max_tip_ratio_interval = max_ratio_interval.loc[max_ratio_interval.groupby('payment_type')['tip_fare_ratio'].idxmax()]
# print(max_tip_ratio_interval.head())

In [ ]:
# QN 12: Find the location with the highest average fare amount
highest_fare_location = data_cleaned.groupby('PULocationID')['fare_amount'].mean().idxmax()
print("\nLocation with the highest average fare amount:")
print(highest_fare_location)

In [ ]:
# QN 13: Build a dataframe for each pickup location and its 5 most common destinations
common_destinations = data_cleaned.groupby('PULocationID')['DOLocationID'].value_counts().groupby(level=0).nlargest(5).reset_index(level=0, drop=True).reset_index()
common_destinations.columns = ['PULocationID', 'DOLocationID', 'trip_count']
print(common_destinations)

In [ ]:
# QN 14: For each payment type and interval in the common dataframe, determine the average fare amount
common_data = data_cleaned.merge(common_destinations, on=['PULocationID', 'DOLocationID'], how='inner')
common_fare_stats = common_data.groupby(['payment_type', 'pickup_interval'])['fare_amount'].mean().reset_index()
print(common_fare_stats)

In [ ]:
# QN 15: Compute the difference of the average fare amount with those computed in point 9
common_fare_stats = common_fare_stats.merge(interval_stats, on='pickup_interval', suffixes=('_common', '_overall'))
common_fare_stats['fare_diff'] = common_fare_stats['fare_amount_common'] - common_fare_stats['fare_amount']
print(common_fare_stats)


In [ ]:
# QN 16: Compute the ratio of the differences to the overall fare amounts
common_fare_stats['fare_diff_ratio'] = common_fare_stats['fare_diff'] / common_fare_stats['fare_amount']
print(common_fare_stats.head())  # Shows the first 5 rows


In [ ]:
# QN 17: Build chains of trips
# Ensure datetime columns are in datetime format
data_cleaned['tpep_pickup_datetime'] = pd.to_datetime(data_cleaned['tpep_pickup_datetime'])
data_cleaned['tpep_dropoff_datetime'] = pd.to_datetime(data_cleaned['tpep_dropoff_datetime'])

# Adding a chain column and identifying chains based on VendorID, locations, and time constraints
data_cleaned = data_cleaned.sort_values(by=['VendorID', 'tpep_pickup_datetime'])
data_cleaned['chain'] = 0
chain_id = 1
for i in range(1, len(data_cleaned)):
    prev_trip = data_cleaned.iloc[i - 1]
    curr_trip = data_cleaned.iloc[i]
    if (
        prev_trip['VendorID'] == curr_trip['VendorID'] and
        prev_trip['DOLocationID'] == curr_trip['PULocationID'] and
        0 < (curr_trip['tpep_pickup_datetime'] - prev_trip['tpep_dropoff_datetime']).total_seconds() <= 120
    ):
        data_cleaned.at[i, 'chain'] = chain_id
    else:
        chain_id += 1
        data_cleaned.at[i, 'chain'] = chain_id
        # print(data_cleaned[['VendorID', 'PULocationID', 'DOLocationID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'chain']])
print(data_cleaned[['VendorID', 'PULocationID', 'DOLocationID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'chain']].head())        
